# COVID-19 Analysis — Example Notebook

Quick demonstrations of the three analyses in this project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Data

In [ ]:
df = pd.read_csv('owid-covid-data.csv', parse_dates=['date'])
print(f'Shape: {df.shape}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Countries: {df["location"].nunique()}')

## 1. Healthcare Strain — ICU vs Lagged Deaths

In [ ]:
country = 'United States'
cdata = df[df['location'] == country].copy()

fig, ax = plt.subplots()
clean = cdata[['icu_patients_per_million', 'new_deaths_smoothed_per_million']].dropna()
if len(clean) > 0:
    ax.scatter(clean['new_deaths_smoothed_per_million'],
               clean['icu_patients_per_million'], alpha=0.4)
    corr = clean.corr().iloc[0, 1]
    ax.set_title(f'{country} — ICU vs Deaths (r={corr:.3f})')
    ax.set_xlabel('Deaths per Million (7-day avg)')
    ax.set_ylabel('ICU Patients per Million')
plt.show()

## 2. Pandemic Fatigue Detection

In [ ]:
cdata['case_14d_avg'] = cdata['new_cases_smoothed_per_million'].rolling(14, min_periods=7).mean()
cdata['case_change'] = cdata['case_14d_avg'].pct_change(periods=14)
cdata['fatigue'] = (cdata['stringency_index'] >= 60) & (cdata['case_change'] > 0.2)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1b = ax1.twinx()
ax1.plot(cdata['date'], cdata['stringency_index'], label='Stringency', color='blue')
ax1b.plot(cdata['date'], cdata['new_cases_smoothed_per_million'], label='Cases/M', color='red', alpha=0.7)
ax1.set_ylabel('Stringency', color='blue')
ax1b.set_ylabel('Cases/M', color='red')
ax1.set_title(f'{country} — Pandemic Fatigue')

ax2.fill_between(cdata['date'], 0, cdata['fatigue'].astype(int), alpha=0.5, color='orange')
ax2.set_ylabel('Fatigue Indicator')
ax2.set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()

print(f'Fatigue days: {cdata["fatigue"].sum()} / {len(cdata)} ({100*cdata["fatigue"].mean():.1f}%)')

## 3. Policy Lag — Cross-Correlation

In [ ]:
valid = cdata[['stringency_index', 'reproduction_rate']].dropna()

lags = range(0, 31)
corrs = []
for lag in lags:
    if lag == 0:
        c = valid['stringency_index'].corr(valid['reproduction_rate'])
    else:
        c = valid['stringency_index'].iloc[:-lag].corr(valid['reproduction_rate'].iloc[lag:])
    corrs.append(c)

best_lag = list(lags)[corrs.index(min(corrs))]

fig, ax = plt.subplots()
colors = ['#EF553B' if c < 0 else '#636EFA' for c in corrs]
ax.bar(lags, corrs, color=colors)
ax.set_xlabel('Lag (days)')
ax.set_ylabel('Correlation')
ax.set_title(f'{country} — Stringency vs R (best lag: {best_lag} days, r={min(corrs):.3f})')
plt.show()

## 4. Multi-Country Comparison

In [ ]:
countries = ['United States', 'United Kingdom', 'Germany', 'France']

fig, ax = plt.subplots()
for c in countries:
    cd = df[df['location'] == c]
    ax.plot(cd['date'], cd['new_cases_smoothed_per_million'], label=c, linewidth=2)

ax.set_xlabel('Date')
ax.set_ylabel('Cases per Million (7-day avg)')
ax.set_title('COVID-19 Cases Comparison')
ax.legend()
plt.tight_layout()
plt.show()